# T3.1 Recursión: conceptos, tipología, pila de llamadas y diseño

## 1. Introducción
Se denomina recursivo a cualquier ente (definición, proceso, estructura, etc.) que se define en función de sí mismo.

En programación, una función es recursiva si su resolución requiere la solución previa de la función para un caso más sencillo. Es decir, ¡si se invoca a sí misma!

**Ejemplo: Función factorial**
$$
\text{factorial}(n) = \begin{cases} 
1 & \text{si } n = 0 \\
n \cdot \text{factorial}(n-1) & \text{si } n > 0 
\end{cases}
$$

Para resolver un problema complejo mediante un algoritmo recursivo, se descompone el problema en una serie de problemas más simples que se resuelven empleando el mismo algoritmo, para luego componer la solución del problema complejo en base a las soluciones de los problemas más simples.

Hay dos elementos vitales en la recursión:
* **Caso directo o base**: el problema es lo suficientemente simple para ser resuelto de manera trivial, sin realizar más llamadas.
* **Caso recursivo o general**: la resolución del problema viene expresada en términos de la resolución de uno o más subproblemas del mismo tipo y más pequeños en alguna medida.

Una vez planteado un algoritmo recursivo es necesario verificar:
* **Terminación**: que en algún momento se llegará al caso base a partir de cualquier caso general planteado inicialmente.
* **Corrección**: que para cualquier subproblema que se dé, la solución del algoritmo es correcta (inducción matemática).

## 2. Diseño de un método recursivo

Los algoritmos recursivos se implementan fácilmente usando funciones en Python. La estructura fundamental consiste en tener una instrucción condicional (`if/else`) que permita distinguir entre el caso base y el caso general.

Siguiendo con el ejemplo del factorial:

In [ ]:
def factorial(n):
    # Precondición: n >= 0
    if n == 0:
        return 1  # Instrucciones del caso base
    else:
        return n * factorial(n - 1)  # Instrucciones del caso general

print(f'El factorial de 3 es: {factorial(3)}')

El factorial de 3 es: 6


Evidentemente, es posible resolver el mismo problema utilizando iteración (bucles `for` o `while`):

In [ ]:
def factorial_iterativo_for(n):
    resultado = 1
    for i in range(1, n + 1):
        resultado *= i
    return resultado

def factorial_iterativo_while(n):
    resultado = 1
    i = 2
    while i <= n:
        resultado *= i
        i += 1
    return resultado

def factorial_iterativo_while_reverse(n):
    resultado = 1
    i = n
    while i > 1:
        resultado *= i
        i -= 1
    return resultado

print(f'El factorial iterativo (for) de 3 es: {factorial_iterativo_for(3)}')
print(f'El factorial iterativo (while) de 3 es: {factorial_iterativo_while(3)}')
print(f'El factorial iterativo (while con recorrido inverso) de 3 es: {factorial_iterativo_while_reverse(3)}')

El factorial iterativo (for) de 3 es: 6
El factorial iterativo (while) de 3 es: 6
El factorial iterativo (while con recorrido inverso) de 3 es: 6


Para el caso concreto de factorial, la solución recursiva es mucho más **natural** (leíble, entendible) que la iterativa. 

_(De hecho, la implementación recursiva es una transcripción literal de la formulación formal matemática)_

En general, y aunque lo discutiremos los matices más adelante en el tema, **preferiremos las soluciones más "naturales" al problema**, en nuestra apuesta clara por **generar código legible y fácil de mantener**. 

> **Aviso Importante**: Como regla general, ¡**los algoritmos recursivos NO usan bucles** (`for` o `while`)! 
> 
> El propósito fundamental de la recursión es utilizar y explotar la pila de memoria del sistema (como ilustraremos más adelante) para realizar la repetición de instrucciones, en sustitución directa de las estructuras de control iterativas.

## 3. Tipos de recursión
* **Recursión Lineal**: hay a lo sumo una sola llamada recursiva en cada ejecución del algoritmo.
  * **Final**: el resultado de la llamada recursiva es el propio resultado de la llamada actual; es decir, la solución del caso más simple es a su vez la del caso más complejo.
  * **No final**: el resultado de la llamada recursiva se emplea para calcular el resultado de la llamada actual, arrojando un resultado posiblemente distinto al de la llamada recursiva. 
* **Recursión Múltiple**: hay más de una llamada recursiva en cada ejecución del algoritmo, y sus resultados han de combinarse. Se genera un árbol de llamadas.

Por ejemplo, `factorial(n)` es una función recursiva **lineal no final**, ya que el resultado de la llamada recursiva es multiplicado por `n`. Este es el esquema de la secuencia de llamadas que genera `factorial(3)`:

```text
factorial(3)
│
├──> factorial(2)
│   │
│   ├──> factorial(1)
│   │   │
│   │   ├──> factorial(0) ────> retorna 1
│   │   │
│   │   └─────────────────────> retorna 1 * 1 = 1
│   │
│   └─────────────────────────> retorna 2 * 1 = 2
│
└─────────────────────────────> retorna 3 * 2 = 6
```

A lo largo del tema iremos viendo diferentes tipos de recursión.

## 4. Recursividad y pila de llamadas (Call Stack)
Cada llamada recursiva a una misma función está asociada a un **registro de activación** (o *stack frame*) propio, donde se guardan las variables locales de dicha llamada y desde dónde debe continuar la ejecución al retornar de la misma. Dichos registros se apilan en la **pila de llamadas**.

Cuando una ejecución de un método finaliza, deja de existir su registro de activación (se desapila), y la ejecución se reanuda en la función que lo llamó, recuperando su propio contexto.

En Python, usar recursividad hace un uso muy intensivo de la memoria, y existe un límite de llamadas recursivas para evitar que se desborde la pila (*Stack Overflow*), provocando que el programa caiga estrepitosamente. Podemos consultar e incluso cambiar este límite en Python mediante el módulo `sys`.

In [ ]:
import sys, inspect

print('Límite de recursión actual en Python:', sys.getrecursionlimit())

#sys.setrecursionlimit(3000) # podemos modificar el límite de recursión

def recursion_infinita(n=1):
    return recursion_infinita(n+1)

try:
    recursion_infinita()
except RecursionError:
    n_max = inspect.trace()[-1][0].f_locals.get('n') # Obtenemos el valor de n en el último registro
    print(f"¡Stack overflow! La pila se ha desbordado, con n = {n_max}.")

Límite de recursión actual en Python: 3000
¡Stack overflow! La pila se ha desbordado, con n = 2978.


### Ejemplo: pila de llamadas para factorial

A continuación, vamos a visualizar una **animación paso a paso** de la ejecución de `factorial(3)`. 

Veamos el código completo de `factorial()` y una función `main()` invocándola, para luego visualizar cómo esto se traduce en la gestión de la memoria del ordenador (pila de llamadas):

In [ ]:
def factorial(n):
    if n == 0:
        ret = 1
    else:
        ret = n * factorial(n - 1)
    return ret

def main():
    print("Inicio del programa main()")
    resultado = factorial(3)
    print(f"El factorial de 3 es: {resultado}")
    print("Fin del programa")

main()

Inicio del programa main()
El factorial de 3 es: 6
Fin del programa


Ejecuta la celda inferior para generar la animación paso a paso donde observaremos cómo los contextos de activación (con sus variables `n` y `ret`) se van apilando y desapilando de la pila de llamadas. Ajusta el tiempo de espera entre cada paso para observar la animación con más detalle.

In [ ]:
import time
from IPython.display import clear_output, display, HTML

def animar_factorial():
    pasos = [
        {'accion': 'El Sistema Operativo llama a main()', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}]},
        {'accion': 'main() llama a f(3)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}]},
        {'accion': 'f(3) llama a f(2)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}, {'func': 'f(2)', 'n': 2, 'r': '?'}]},
        {'accion': 'f(2) llama a f(1)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}, {'func': 'f(2)', 'n': 2, 'r': '?'}, {'func': 'f(1)', 'n': 1, 'r': '?'}]},
        {'accion': 'f(1) llama a f(0)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}, {'func': 'f(2)', 'n': 2, 'r': '?'}, {'func': 'f(1)', 'n': 1, 'r': '?'}, {'func': 'f(0)', 'n': 0, 'r': '?'}]},
        {'accion': 'f(0) alcanza el caso base', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}, {'func': 'f(2)', 'n': 2, 'r': '?'}, {'func': 'f(1)', 'n': 1, 'r': '?'}, {'func': 'f(0)', 'n': 0, 'r': 1}]},
        {'accion': 'f(0) retorna 1 a f(1)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}, {'func': 'f(2)', 'n': 2, 'r': '?'}, {'func': 'f(1)', 'n': 1, 'r': '1 * 1 = 1'}]},
        {'accion': 'f(1) retorna 1 a f(2)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '?'}, {'func': 'f(2)', 'n': 2, 'r': '2 * 1 = 2'}]},
        {'accion': 'f(2) retorna 2 a f(3)', 'pila': [{'func': 'main()', 'n': '-', 'r': '?'}, {'func': 'f(3)', 'n': 3, 'r': '3 * 2 = 6'}]},
        {'accion': 'f(3) retorna 6 a main()', 'pila': [{'func': 'main()', 'n': '-', 'r': '?', 'resultado': 6}]},
        {'accion': 'main() termina y la pila se vacía', 'pila': []}
    ]
    
    for i, paso in enumerate(pasos):
        clear_output(wait=True)
        html = f"<h3>Paso {i+1}: <span style=''>{paso['accion']}</span></h3>"
        html += "<div style='display: flex; flex-direction: column; width: 380px; border: 2px solid #333; padding: 10px; background-color: #f4f6f9; border-radius: 8px;'>"
        
        if not paso['pila']:
            html += "<div style='text-align: center; color: #888; font-style: italic; padding: 20px;'>Pila de llamadas vacía</div>"
            
        # Invertimos la lista para pintar los registros apilados como en la memoria (último en entrar, arriba)
        for p in reversed(paso['pila']):
            # Color especial para main()
            bg_color = "#e9f2fe" if p['func'] != 'main()' else "#e2f0d9"
            border_color = "#007bff" if p['func'] != 'main()' else "#28a745"
            
            html += f"<div style='border: 1px solid {border_color}; margin: 5px 0; padding: 10px; background-color: {bg_color}; border-radius: 5px; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);'>"
            html += f"<h4 style='margin: 0 0 5px 0; color: {border_color};'>Registro: {p['func']}</h4>"
            if p['func'] != 'main()':
                html += f"<span><code>n = <b>{p['n']}</b></code></span><br>"
                html += f"<span><code>ret = <b>{p['r']}</b></code></span>"
            else:
                html += f"<span><code>resultado = <b>{p.get('resultado', '?')}</b></code></span><br>"
            html += "</div>"
            
        html += "</div>"
        display(HTML(html))
        time.sleep(PAUSA_ENTRE_PASOS)
        
    print("¡Fin de la ejecución del programa!")

# Descomenta y ejecuta la siguiente línea para reproducir la animación:
PAUSA_ENTRE_PASOS = 3 # segundos, ajusta según tu preferencia
animar_factorial()

¡Fin de la ejecución del programa!


## 5. Etapas del diseño recursivo

El diseño recursivo consta de cuatro etapas:

#### 1. **Enunciado del problema**: 

Entender el problema y establecer el enunciado formal:
- Declarar la cabecera o perfil de la función (nombre de la función, parámetros, retorno).
- Establecer las condiciones de los parámetros de entrada (restricciones).
- Establecer el resultado esperado.

Las condiciones y el resultado se suelen reflejar en un comentario multilinea antes de la cabecera de la función.

#### 2. **Análisis de casos**: 

Identificar conceptualmente el caso base y el caso general, y establecer las acciones a ejecutar en cada caso.
- Notar que en ciertos problemas puede haber más de un caso base.
- Hay que comprobar que se cubre cualquier caso posible.

#### 3. **Transcripción**: 

Transcribir los casos y las acciones a realizar en forma de algoritmo recursivo.

#### 4. **Validación**: 

Determinar que, en cada nueva llamada recursiva:
  + el nuevo problema a resolver es *estrictamente más cercano al caso base*.
  + las sucesivas llamadas recursivas cumplen con las condiciones de entrada.


**Ejemplo aplicado al Factorial:**
1. **Enunciado del problema**: 

    ```python
    """
    Devuelve el factorial de un número entero no negativo.
    
    Parámetros:
        n (int): Un número entero no negativo (n >= 0).
    
    Retorna:
        int: El factorial de n.
    """
    def factorial(n):
   ```
2. **Análisis de casos**: 
   - *Caso base*: `n == 0` $\rightarrow$ El factorial es 1.
   - *Caso general*: `n > 0` $\rightarrow$ El factorial es $n \cdot \text{factorial}(n-1)$.

3. **Transcripción**: 

    El código Python formal que usa `if n == 0: return 1` y `else: return n * factorial(n-1)`.

4. **Validación**: 
    - En la llamada `factorial(n-1)`, el parámetro es estrictamente menor, acercándose hacia 0. Como `n` es entero positivo, alcanzará obligatoriamente el caso base `0`, asegurando la terminación.
    - En toda llamada subsiguiente, se garantiza que $n \ge 0$. 